In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
dim_products = spark.read.table("databricksintech.silver.product")
dim_products.limit(3).display()

Product_id,Product_name,Product_number,Color,Standard_cost,List_price,Size,Weight,Product_category_id,Product_model_id,Sell_start_date,Sell_end_date,Discontinued_date,Modified_date
680,"HL Road Frame - Black, 58",FR-R92B-58,Black,1059.31,1431.5,58,1016.04,18,6,2002-06-01,null,null,2008-03-11
706,"HL Road Frame - Red, 58",FR-R92R-58,Red,1059.31,1431.5,58,1016.04,18,6,2002-06-01,null,null,2008-03-11
707,"Sport-100 Helmet, Red",HL-U509-R,Red,13.0863,34.99,null,null,35,33,2005-07-01,null,null,2008-03-11


In [0]:
order = spark.read.table("databricksintech.silver.order_detail")

In [0]:
order = order.withColumn('Total_order_price', col('Order_qty') * \
                        col("Unit_price") * (1 - col("Unit_price_discount")))\
                        .withColumnRenamed('Product_id', 'Product_id2')

order = order.groupby("Product_id2").agg(sum("Total_order_price").alias("Total_amount_sold"), sum("Order_qty").alias("Total_qty_sold"))


In [0]:
dim_product = dim_products.join(order, dim_products.Product_id==order.Product_id2, 'left')
dim_product = dim_product.drop("Product_id2")
dim_product.limit(3).display()

Product_id,Product_name,Product_number,Color,Standard_cost,List_price,Size,Weight,Product_category_id,Product_model_id,Sell_start_date,Sell_end_date,Discontinued_date,Modified_date,Total_amount_sold,Total_qty_sold
680,"HL Road Frame - Black, 58",FR-R92B-58,Black,1059.31,1431.5,58,1016.04,18,6,2002-06-01,null,null,2008-03-11,null,null
706,"HL Road Frame - Red, 58",FR-R92R-58,Red,1059.31,1431.5,58,1016.04,18,6,2002-06-01,null,null,2008-03-11,null,null
707,"Sport-100 Helmet, Red",HL-U509-R,Red,13.0863,34.99,null,null,35,33,2005-07-01,null,null,2008-03-11,734.7899971008301,35.0


In [0]:
(
    dim_product.write.format("delta")
    .mode("append") 
    .option(
        "path",
        "abfss://gold@intechaccstorage.dfs.core.windows.net/dim_products",
    )
    .saveAsTable("databricksintech.gold.dim_products")
)

In [0]:
# Reference target Delta table
gold_table = DeltaTable.forName(spark, "databricksintech.gold.dim_products")

# Execute the Merge (Upsert)
gold_table.alias("target").merge(
    source = dim_product.alias("source"),
    condition = (
        "target.Product_id = source.Product_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
